# Auswertung und Analyse der LLM Evaluationsergebnisse

Dieses Notebook führt eine umfassende Analyse der LLM-Auswertungsergebnisse durch. Es werden die Daten eingelesen, inspiziert, bereinigt, statistisch ausgewertet und visualisiert.

## 1. Daten einlesen
Wir lesen die Auswertungsdaten aus der CSV-Datei mit pandas ein.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Robustes Laden der CSV (funktioniert aus Notebook-Ordner oder Projekt-Root)
base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
candidates = [
    base / 'evaluation_results_export.csv',
    base / '../results/evaluation_results_export.csv',
    base / '../../results/evaluation_results_export.csv',
    base.parent / 'results/evaluation_results_export.csv',
    base.parent / 'evaluation/results/evaluation_results_export.csv',
    base.parent.parent / 'evaluation/results/evaluation_results_export.csv',
    Path('evaluation/results/evaluation_results_export.csv'),
    Path('results/evaluation_results_export.csv')
]
csv_path = next((p.resolve() for p in candidates if p.exists()), None)

if csv_path is None:
    tried = '\n'.join(str(p.resolve()) for p in candidates)
    raise FileNotFoundError(f'CSV nicht gefunden. Versuch folgende Pfade:\n{tried}')

df = pd.read_csv(csv_path)
print(f'Datei geladen: {csv_path}, Zeilen: {len(df)}')
df.head()

Datei geladen: /Users/frederikbrinkmann/Documents/CODING/LLM-Pipeline/LLM-Adapter-Pipeline/evaluation/results/evaluation_results_export.csv, Zeilen: 1400


,model_id,email_id,timestamp,status,schema_valid,field_accuracy,critical_field_accuracy,time_ms,summary,description,claim_type,claim_amount,priority,policy_number,claim_date,incident_date,incident_location,missing_fields,error
0,gpt-5.2,EMAIL_GEN_001,2026-03-01T12:38:37.222675,success,True,100.0,92.3,9555.8,"Wasserschaden durch Rohrbruch in Mietwohnung, ...",Meldung eines Wasserschadens in einer Mietwohn...,damage,18500,high,POL-2025-4187,NaN,2026-02-03,"Rosenstraße 14, 2. OG, 60329 Frankfurt am Main",claim_date,NaN
1,gpt-5.2,EMAIL_GEN_002,2026-03-01T12:38:47.090404,success,True,100.0,84.6,9871.1,Wasserschaden in Mietwohnung am 2026-02-14 (ve...,Der Versicherungsnehmer meldet einen Wassersch...,damage,NaN,high,POL-2025-4831,NaN,2026-02-14,"Wohnung (Mietwohnung), Bereich Küche/Bad; Wass...","claimant_email,claim_date",NaN
2,gpt-5.2,EMAIL_GEN_003,2026-03-01T12:38:57.232420,success,True,92.3,92.3,9076.1,Wasserschaden in Wohnung (vermutlich Rohrbruch...,Der Versicherungsnehmer meldet einen Wassersch...,damage,NaN,high,POL-2025-4187,NaN,NaN,Wohnung des Versicherungsnehmers (genaue Adres...,"claimant_email,claim_date,incident_date",NaN
3,gpt-5.2,EMAIL_GEN_004,2026-03-01T12:39:06.462126,success,True,100.0,92.3,10459.1,Wasserschaden in Wohnung durch vermuteten Rohr...,Der Versicherungsnehmer meldet einen deutliche...,damage,NaN,high,POL-2025-4187,NaN,NaN,NaN,"claimant_email,claim_date,incident_date,incide...",NaN
4,gpt-5.2,EMAIL_GEN_005,2026-03-01T12:39:17.118661,success,True,84.6,76.9,9121.9,Rohrbruch Küche mit Wasserschaden in Wohnung (...,Der Versicherungsnehmer Martin Keller meldet e...,damage,25000,high,POL-2025-4187,2026-02-15,2026-02-15,"Lindenstraße 12, 40233 Düsseldorf",claimant_email,NaN


In [ ]:
# Normalisierung: Modellspalte und Feldspalten-Fallbacks
model_candidates = ['model', 'model_id', 'model_name', 'provider_model', 'llm_model']
model_col = next((c for c in model_candidates if c in df.columns), None)
if model_col and model_col != 'model':
    df['model'] = df[model_col]
    print(f'Modellspalte gesetzt von {model_col} -> model')
elif model_col == 'model':
    print('Modellspalte vorhanden: model')
else:
    print(f'Keine Modellspalte gefunden; Kandidaten: {model_candidates}')

field_cols_candidates = [c for c in df.columns if any(p in c for p in ['field_accuracy_', 'exact_match_', 'field_score_', 'accuracy_'])]
if field_cols_candidates:
    print('Feld-Genauigkeit/Score-Spalten erkannt:', field_cols_candidates)
else:
    print('Keine Feldgenauigkeits-/Score-Felder erkannt (field_accuracy_, exact_match_, field_score_, accuracy_).')

## 2. Erste Dateninspektion
Wir verschaffen uns einen Überblick über die Datenstruktur, Spalten, Datentypen und prüfen auf fehlende Werte.

In [33]:
# Schneller Diagnose-Check nach dem Laden
print('Zeilen x Spalten:', df.shape)
print('Spalten:', list(df.columns))
print('Anzahl numerischer Spalten:', len(df.select_dtypes(include=[np.number]).columns))
print('Anzahl object-Spalten:', len(df.select_dtypes(include=['object']).columns))

score_cols = [col for col in df.columns if 'score' in col or 'similarity' in col]
field_cols = [c for c in df.columns if c.startswith('field_accuracy_') or c.startswith('exact_match_')]
print('Score-Spalten:', score_cols)
print('Feld-Genauigkeitsspalten:', field_cols)
print('Model-Spalte vorhanden:', 'model' in df.columns)
if 'model' in df.columns:
    print('Modelle:', df['model'].unique())

Zeilen x Spalten: (1400, 19)
Spalten: ['model_id', 'email_id', 'timestamp', 'status', 'schema_valid', 'field_accuracy', 'critical_field_accuracy', 'time_ms', 'summary', 'description', 'claim_type', 'claim_amount', 'priority', 'policy_number', 'claim_date', 'incident_date', 'incident_location', 'missing_fields', 'error']
Anzahl numerischer Spalten: 3
Anzahl object-Spalten: 16
Score-Spalten: []
Feld-Genauigkeitsspalten: []
Model-Spalte vorhanden: False


/var/folders/lp/8kwttmj10_v9vjg5dp4m64ym0000gn/T/ipykernel_59099/4193872594.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print('Anzahl object-Spalten:', len(df.select_dtypes(include=['object']).columns))


In [34]:
# Überblick über die Spalten und Datentypen
print('Spalten:', df.columns.tolist())
df.info()

# Fehlende Werte pro Spalte
print('\nFehlende Werte pro Spalte:')
display(df.isnull().sum())

# Ein paar Beispielzeilen anzeigen
df.sample(5)

Spalten: ['model_id', 'email_id', 'timestamp', 'status', 'schema_valid', 'field_accuracy', 'critical_field_accuracy', 'time_ms', 'summary', 'description', 'claim_type', 'claim_amount', 'priority', 'policy_number', 'claim_date', 'incident_date', 'incident_location', 'missing_fields', 'error']
<class 'pandas.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model_id                 1400 non-null   str    
 1   email_id                 1400 non-null   str    
 2   timestamp                1400 non-null   str    
 3   status                   1400 non-null   str    
 4   schema_valid             990 non-null    object 
 5   field_accuracy           990 non-null    float64
 6   critical_field_accuracy  990 non-null    float64
 7   time_ms                  990 non-null    float64
 8   summary                  990 non-null    str    
 9   description 

model_id                      0
email_id                      0
timestamp                     0
status                        0
schema_valid                410
field_accuracy              410
critical_field_accuracy     410
time_ms                     410
summary                     410
description                 420
claim_type                  411
claim_amount               1164
priority                    411
policy_number               428
claim_date                 1303
incident_date               902
incident_location           871
missing_fields              476
error                       990
dtype: int64

,model_id,email_id,timestamp,status,schema_valid,field_accuracy,critical_field_accuracy,time_ms,summary,description,claim_type,claim_amount,priority,policy_number,claim_date,incident_date,incident_location,missing_fields,error
656,o4-mini,EMAIL_GEN_057,2026-03-01T23:31:56.570607,success,True,92.3,76.9,13790.6,Medical claim for Daniel Krüger: appendectomy ...,Attorney Katharina Seidel reported a medical c...,medical,8746.3,urgent,POL-2025-4187,NaN,2026-01-18,St. Marien-Klinikum Köln,"subject,claimant_email,claim_date",NaN
664,o4-mini,EMAIL_GEN_065,2026-03-01T23:33:26.993308,success,True,76.9,76.9,12036.7,Storm damage: hail roof damage in Rüsselsheim ...,Hagelschaden am Dach mit mehreren kaputten Zie...,damage,18000,high,POL-2025-4831,NaN,2026-02-15,"Gartenstraße 12, 65428 Rüsselsheim",claim_date,NaN
555,o3,EMAIL_GEN_056,2026-03-01T23:07:15.637379,success,True,76.9,76.9,13269.6,Emergency hospitalization claim - Markus Neumann,Markus Neumann reports a recent serious medica...,medical,NaN,urgent,POL-2025-4187,NaN,NaN,NaN,"claimant_email,claim_date,incident_date",NaN
1237,phi3:mini,EMAIL_GEN_038,2026-03-02T14:37:27.824246,success,True,76.9,76.9,20364.3,"Multi-vehicle collision in Köln, Kfz insurance...",Multi-vehicle collision in Köln on February 13...,damage,NaN,medium,POL-2025-4187,NaN,2026-02-13,Köln,description,NaN
504,o3,EMAIL_GEN_005,2026-03-01T22:57:37.634339,success,True,76.9,76.9,12775.5,Burst kitchen pipe causing extensive water dam...,Claimant reports that on 15.02.2026 a pipe bur...,damage,25000,high,POL-2025-4187,2026-02-15,2026-02-15,"Lindenstraße 12, 40233 Düsseldorf",NaN,NaN


## 3. Datenbereinigung
Wir bereinigen die Daten, indem wir fehlende Werte behandeln und Datentypen anpassen, falls nötig.

In [24]:
# Beispiel: Fehlende Werte auffüllen oder Zeilen entfernen
# (Hier kann je nach Analyseziel angepasst werden)

# Zunächst: Übersicht der fehlenden Werte
missing = df.isnull().sum()
print(missing[missing > 0])

# Kopie, damit keine SettingWithCopy-Warnungen kommen
df_clean = df.copy()

# Beispiel: Entferne Zeilen mit zu vielen fehlenden Werten
df_clean = df_clean.dropna(thresh=int(0.8*len(df_clean.columns)))

# Beispiel: Fülle numerische Spalten mit Median auf
num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# Beispiel: Fülle kategorische Spalten mit Modus auf
cat_cols = df_clean.select_dtypes(include=['object']).columns
if len(cat_cols) > 0:
    df_clean[cat_cols] = df_clean[cat_cols].fillna(df_clean[cat_cols].mode().iloc[0])

print('Bereinigte Form:', df_clean.shape)
df_clean.info()

schema_valid                410
field_accuracy              410
critical_field_accuracy     410
time_ms                     410
summary                     410
description                 420
claim_type                  411
claim_amount               1164
priority                    411
policy_number               428
claim_date                 1303
incident_date               902
incident_location           871
missing_fields              476
error                       990
dtype: int64
Bereinigte Form: (624, 19)
<class 'pandas.DataFrame'>
Index: 624 entries, 0 to 1397
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model_id                 624 non-null    str    
 1   email_id                 624 non-null    str    
 2   timestamp                624 non-null    str    
 3   status                   624 non-null    str    
 4   schema_valid             624 non-null    object 
 5   field_a

/var/folders/lp/8kwttmj10_v9vjg5dp4m64ym0000gn/T/ipykernel_59099/2725521238.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_clean.select_dtypes(include=['object']).columns


## 4. Deskriptive Statistiken
Wir berechnen zentrale Kennzahlen wie Mittelwert, Median, Standardabweichung und weitere für relevante numerische Spalten.

### Feldgenauigkeit pro Modell (sofern vorhanden)
Falls es Spalten wie `field_accuracy_*` oder `exact_match_*` gibt, werden diese ebenfalls pro Modell aggregiert.

In [35]:
# Feldgenauigkeiten pro Modell aggregieren (Spalten, die mit field_accuracy_ oder exact_match_ beginnen)
field_cols = [c for c in df_clean.columns if c.startswith('field_accuracy_') or c.startswith('exact_match_')]
if 'model' in df_clean.columns and field_cols:
    agg_fields = df_clean.groupby('model')[field_cols].mean().round(4)
    display(agg_fields)
else:
    print('Keine Feld-Genauigkeitsspalten gefunden oder Spalte "model" fehlt.')

Keine Feld-Genauigkeitsspalten gefunden oder Spalte "model" fehlt.


In [36]:
# Deskriptive Statistiken für numerische Spalten
if 'df_clean' not in globals():
    df_clean = df.copy()

stats_num = df_clean.describe().T
print('Deskriptive Statistiken (alle numerischen Spalten):')
display(stats_num)

Deskriptive Statistiken (alle numerischen Spalten):


,count,mean,std,min,25%,50%,75%,max
field_accuracy,624.0,76.443269,11.152343,38.5,69.20,76.9,84.6,100.0
critical_field_accuracy,624.0,72.851763,10.015559,38.5,69.20,76.9,76.9,92.3
time_ms,624.0,35543.213462,29888.406615,5247.1,12586.05,23535.4,49970.4,118945.4


In [37]:
# Statistiken für Score-Spalten (falls vorhanden)
if 'df_clean' not in globals():
    df_clean = df.copy()

score_cols = [col for col in df_clean.columns if 'score' in col or 'similarity' in col]
if score_cols:
    stats_scores = df_clean[score_cols].describe().T
    print('Score-Spalten erkannt:', score_cols)
    display(stats_scores)
else:
    print('Keine Score-Spalten gefunden.')

Keine Score-Spalten gefunden.


## 5. Visualisierung der Daten
Wir visualisieren die wichtigsten Metriken, z.B. Score-Verteilungen, Modellvergleiche und Fehlerarten.

In [38]:
# Beispiel: Score-Verteilung für ein Modell
if score_cols:
    for col in score_cols:
        plt.figure(figsize=(8,4))
        sns.histplot(df_clean[col], kde=True, bins=20)
        plt.title(f'Verteilung: {col}')
        plt.xlabel(col)
        plt.ylabel('Anzahl')
        plt.show()
else:
    print('Keine Score-Spalten für Visualisierung gefunden.')

Keine Score-Spalten für Visualisierung gefunden.


In [14]:
# Beispiel: Boxplot für Score-Spalten nach Modell (falls Spalte "model" vorhanden)
if 'model' in df_clean.columns and score_cols:
    for col in score_cols:
        plt.figure(figsize=(10,5))
        sns.boxplot(x='model', y=col, data=df_clean)
        plt.title(f'Boxplot: {col} nach Modell')
        plt.xlabel('Modell')
        plt.ylabel(col)
        plt.xticks(rotation=45)
        plt.show()

## 5a. Per-Modell-Auswertung (Scores)
Aggregierte Kennzahlen je Modell (mean/median/std/count) und Ranking nach dem ersten Score-Feld.

In [39]:
# Aggregierte Kennzahlen pro Modell (Scores)
if 'model' in df_clean.columns:
    if score_cols:
        agg = df_clean.groupby('model')[score_cols].agg(['mean','median','std','count']).round(4)
        display(agg)
        primary = score_cols[0]
        ranking = df_clean.groupby('model')[primary].mean().sort_values(ascending=False)
        print('\nRanking nach Mittelwert von', primary)
        display(ranking)
    else:
        print('Keine Score-Spalten gefunden; gruppierte Score-Auswertung entfällt.')
else:
    print('Spalte "model" nicht vorhanden; per-Modell-Auswertung entfällt.')

Spalte "model" nicht vorhanden; per-Modell-Auswertung entfällt.


In [40]:
# Balkenplot: Mittelwert ± SD je Score und Modell
if 'model' in df_clean.columns and score_cols:
    for col in score_cols:
        plt.figure(figsize=(8,4))
        sns.barplot(x='model', y=col, data=df_clean, estimator=np.mean, ci='sd')
        plt.title(f'Mittelwert ± SD: {col} nach Modell')
        plt.xlabel('Modell')
        plt.ylabel(col)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 5b. Latenz & Fehlerquoten
Übersicht über Latenz-Verteilungen je Modell und Anteile fehlender/kritischer Felder oder Fehlerstatus.

## 5c. Feld-Genauigkeit Heatmap
Visualisiert die mittlere Feldgenauigkeit je Modell (falls `field_accuracy_*`/`exact_match_*` vorhanden).

## 5d. Top-/Flop-Beispiele pro Modell
Zeigt je Modell die höchsten/niedrigsten Scores (für schnellen Drilldown).

In [41]:
# Top-/Flop-Beispiele pro Modell
if 'model' in df_clean.columns and score_cols:
    primary = score_cols[0]
    id_cols = [c for c in ['id','email_id','job_id','sample_id'] if c in df_clean.columns]
    cols_to_show = id_cols + ['model', primary]
    n = 3
    print(f'Top {n} und Flop {n} je Modell basierend auf {primary}')
    rows = []
    for m, group in df_clean.groupby('model'):
        top = group.nlargest(n, primary)[cols_to_show].assign(kind='top') if not group.empty else None
        flop = group.nsmallest(n, primary)[cols_to_show].assign(kind='flop') if not group.empty else None
        if top is not None:
            rows.append(top)
        if flop is not None:
            rows.append(flop)
    if rows:
        display(pd.concat(rows))
    else:
        print('Keine Daten für Top/Flop-Auswahl gefunden.')
else:
    print('Kein Modell- oder Score-Feld verfügbar für Top/Flop-Beispiele.')

Kein Modell- oder Score-Feld verfügbar für Top/Flop-Beispiele.


In [42]:
# Feld-Genauigkeit als Heatmap
field_cols = [c for c in df_clean.columns if c.startswith('field_accuracy_') or c.startswith('exact_match_')]
if 'model' in df_clean.columns and field_cols:
    mat = df_clean.groupby('model')[field_cols].mean()
    plt.figure(figsize=(min(12, 2 + len(field_cols)*0.4), 6))
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='Greens')
    plt.title('Feldgenauigkeit je Modell')
    plt.xlabel('Feld')
    plt.ylabel('Modell')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Feld-Genauigkeitsspalten gefunden oder Spalte "model" fehlt.')

Keine Feld-Genauigkeitsspalten gefunden oder Spalte "model" fehlt.


In [ ]:
# Latenz-Analyse und Fehlerquoten
latency_cols = [c for c in df_clean.columns if 'latency' in c.lower()]
if 'model' in df_clean.columns and latency_cols:
    for col in latency_cols:
        plt.figure(figsize=(8,4))
        sns.boxplot(x='model', y=col, data=df_clean)
        plt.title(f'Latenz nach Modell: {col}')
        plt.xlabel('Modell')
        plt.ylabel(col)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    print('Latenz-Spalten:', latency_cols)
    display(df_clean.groupby('model')[latency_cols].describe().round(3))
else:
    print('Keine Latenz-Spalten oder kein Modell-Feld gefunden.')

# Fehler-/Missing-Quoten pro Modell
flags = []
for candidate in ['has_missing_critical_fields', 'has_missing_fields', 'failed', 'error', 'success']:
    if candidate in df_clean.columns:
        flags.append(candidate)
if 'status' in df_clean.columns:
    flags.append('status')
if 'error_message' in df_clean.columns:
    flags.append('error_message')

if 'model' in df_clean.columns and flags:
    summary = {}
    for m, group in df_clean.groupby('model'):
        entry = {}
        if 'has_missing_critical_fields' in group:
            entry['missing_critical_rate'] = group['has_missing_critical_fields'].mean()
        if 'has_missing_fields' in group:
            entry['missing_any_rate'] = group['has_missing_fields'].mean()
        if 'status' in group:
            entry['status_counts'] = group['status'].value_counts(normalize=True).to_dict()
        if 'error_message' in group:
            err_rate = group['error_message'].notna().mean()
            entry['error_message_rate'] = err_rate
        summary[m] = entry
    print('Fehler-/Missing-Quoten je Modell:')
    display(pd.DataFrame(summary).T)
else:
    print('Keine Fehler-/Missing-Flags gefunden oder Spalte "model" fehlt.')

## 6. Korrelationsanalyse
Wir berechnen und visualisieren die Korrelationen zwischen numerischen Variablen.

In [ ]:
# Korrelationsmatrix für numerische Spalten
corr = df_clean.corr(numeric_only=True)
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Korrelationsmatrix')
plt.show()

## 7. Ergebnisse exportieren
Wir speichern die bereinigten Daten und ggf. berechnete Kennzahlen in eine neue Datei.

In [15]:
# Exportiere die bereinigten Daten als neue CSV-Datei
df_clean.to_csv('../../results/evaluation_results_cleaned.csv', index=False)
print('Bereinigte Daten wurden gespeichert.')

OSError: Cannot save file into a non-existent directory: '../../results'